## 1 -  Importing Libraries

In [18]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from collections import Counter

## 2 - Reading Data

In [19]:
data = pd.read_parquet('C:/1 - MyData/1 - Study Santha/z - Projects/BuyProof (AI Product Analyzer)/BuyProof/data/small_data.parquet')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   rating             300000 non-null  float64
 1   review_title       300000 non-null  object 
 2   review_text        300000 non-null  object 
 3   parent_asin        300000 non-null  object 
 4   timestamp          300000 non-null  int64  
 5   helpful_vote       300000 non-null  int64  
 6   verified_purchase  300000 non-null  bool   
 7   category           300000 non-null  object 
 8   product_title      225974 non-null  object 
 9   average_rating     229457 non-null  float64
 10  rating_number      229171 non-null  float64
 11  features           225974 non-null  object 
 12  store              223283 non-null  object 
dtypes: bool(1), float64(3), int64(2), object(7)
memory usage: 27.8+ MB


# 3 - Basic EDA

### 3.1 Info about the data

In [20]:
data.shape
data.dtypes
data.info()
print(data.isnull().sum())
print(data.isnull().mean() * 100)   # % missing per column
data.duplicated(subset=data.columns.difference(['features'])).sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   rating             300000 non-null  float64
 1   review_title       300000 non-null  object 
 2   review_text        300000 non-null  object 
 3   parent_asin        300000 non-null  object 
 4   timestamp          300000 non-null  int64  
 5   helpful_vote       300000 non-null  int64  
 6   verified_purchase  300000 non-null  bool   
 7   category           300000 non-null  object 
 8   product_title      225974 non-null  object 
 9   average_rating     229457 non-null  float64
 10  rating_number      229171 non-null  float64
 11  features           225974 non-null  object 
 12  store              223283 non-null  object 
dtypes: bool(1), float64(3), int64(2), object(7)
memory usage: 27.8+ MB
rating                   0
review_title             0
review_text              0
p

np.int64(284)

In [21]:
data['rating'].value_counts().sort_index()
data['rating'].describe()

count    300000.00000
mean          4.31521
std           1.16812
min           1.00000
25%           4.00000
50%           5.00000
75%           5.00000
max           5.00000
Name: rating, dtype: float64

In [22]:
print(data['review_text'].str.len().describe())
data['review_text'].isnull().sum()

count    300000.000000
mean        343.716887
std         576.435847
min           0.000000
25%          59.000000
50%         159.000000
75%         394.000000
max       23991.000000
Name: review_text, dtype: float64


np.int64(0)

In [23]:
print(data['category'].nunique())
print(data['category'].value_counts())

30
category
Beauty_and_Personal_Care       10000
Gift_Cards                     10000
Patio_Lawn_and_Garden          10000
Cell_Phones_and_Accessories    10000
Toys_and_Games                 10000
Baby_Products                  10000
Industrial_and_Scientific      10000
Subscription_Boxes             10000
Clothing_Shoes_and_Jewelry     10000
Appliances                     10000
Tools_and_Home_Improvement     10000
Arts_Crafts_and_Sewing         10000
Pet_Supplies                   10000
Handmade_Products              10000
Automotive                     10000
Amazon_Fashion                 10000
Electronics                    10000
Sports_and_Outdoors            10000
CDs_and_Vinyl                  10000
Movies_and_TV                  10000
Grocery_and_Gourmet_Food       10000
Health_and_Household           10000
Office_Products                10000
Kindle_Store                   10000
All_Beauty                     10000
Home_and_Kitchen               10000
Musical_Instruments       

In [24]:
print(data['verified_purchase'].value_counts(normalize=True))

verified_purchase
True     0.727773
False    0.272227
Name: proportion, dtype: float64


In [25]:
print( data['features'].apply(type).value_counts())
print( data['timestamp'].dtype)

features
<class 'numpy.ndarray'>    225974
<class 'NoneType'>          74026
Name: count, dtype: int64
int64


In [26]:
print(  data['verified_purchase'].value_counts(normalize=True))

verified_purchase
True     0.727773
False    0.272227
Name: proportion, dtype: float64


In [27]:
print(data[['helpful_vote', 'rating_number']].describe())
print( data['helpful_vote'].value_counts().head(10))

        helpful_vote  rating_number
count  300000.000000   2.291710e+05
mean        1.530087   1.129067e+04
std        11.265101   5.677798e+04
min         0.000000   1.000000e+00
25%         0.000000   8.300000e+01
50%         0.000000   5.900000e+02
75%         1.000000   3.790000e+03
max      1450.000000   1.898759e+06
helpful_vote
0    210306
1     41902
2     15459
3      8378
4      5074
5      3414
6      2453
7      1831
8      1455
9      1108
Name: count, dtype: int64


In [28]:
data[['rating', 'average_rating', 'rating_number', 'helpful_vote']].corr()

,rating,average_rating,rating_number,helpful_vote
rating,1.000000,0.307389,0.017411,-0.036948
average_rating,0.307389,1.000000,0.089106,-0.033827
rating_number,0.017411,0.089106,1.000000,-0.006830
helpful_vote,-0.036948,-0.033827,-0.006830,1.000000


In [29]:
print(data.groupby('verified_purchase')['rating'].mean())
print(data.groupby('category')['rating'].mean().sort_values())

verified_purchase
False    4.323358
True     4.312162
Name: rating, dtype: float64
category
Subscription_Boxes             3.7819
Software                       3.8480
Amazon_Fashion                 4.1072
All_Beauty                     4.1183
Health_and_Personal_Care       4.1453
Beauty_and_Personal_Care       4.1662
Cell_Phones_and_Accessories    4.1910
Pet_Supplies                   4.2306
Health_and_Household           4.2413
Patio_Lawn_and_Garden          4.2561
Movies_and_TV                  4.2646
Grocery_and_Gourmet_Food       4.2765
Automotive                     4.3128
Electronics                    4.3168
Clothing_Shoes_and_Jewelry     4.3582
Books                          4.3591
Appliances                     4.3777
Baby_Products                  4.3789
Kindle_Store                   4.3863
Tools_and_Home_Improvement     4.3959
Sports_and_Outdoors            4.4029
Industrial_and_Scientific      4.4089
Home_and_Kitchen               4.4136
Musical_Instruments            4.4

### 3.2 Data Cleaning

#### Removing Duplicates

In [30]:
data = data.drop_duplicates(
    subset=data.columns.difference(['features'])
)

#### Filling Missing values using custom strategy for each col

In [31]:
def clean_data(data):
    from collections import Counter

    def first_non_null(series):
        s = series.dropna()
        return s.iloc[0] if len(s) else None

    def mean_non_null(series):
        s = series.dropna()
        return s.mean() if len(s) else None

    def mode_or_median(series):
        s = series.dropna()
        if len(s) == 0:
            return None
        counts = Counter(s)
        max_freq = max(counts.values())
        modes = [k for k, c in counts.items() if c == max_freq]
        return modes[0] if len(modes) == 1 else s.median()

    def longest_non_null(series):
        s = series.dropna()
        if len(s) == 0:
            return None
        return max(s, key=lambda x: len(str(x)))

    def mode_only(series):
        s = series.dropna()
        if len(s) == 0:
            return None
        return Counter(s).most_common(1)[0][0]

    # Resolve canonical value per parent_asin
    resolved = data.groupby('parent_asin').agg(
        product_title=('product_title', first_non_null),
        average_rating=('average_rating', mean_non_null),
        rating_number=('rating_number', mode_or_median),
        features=('features', longest_non_null),
        store=('store', mode_only)
    )

    # Fill nulls in data using resolved values
    for col in ['product_title', 'average_rating', 'rating_number', 'features', 'store']:
        fill_map = resolved[col]
        data[col] = data[col].fillna(data['parent_asin'].map(fill_map))

    return data

In [32]:
data = clean_data(data)
print(data.isna().sum()) # sanity check remaining nulls

rating                   0
review_title             0
review_text              0
parent_asin              0
timestamp                0
helpful_vote             0
verified_purchase        0
category                 0
product_title        74007
average_rating       70524
rating_number        70810
features             74007
store                76695
dtype: int64


#### Dropping Null Values

In [37]:
data = data.dropna(subset=['product_title', 'average_rating', 'rating_number', 'features', 'store'])
data = data.reset_index(drop=True)

print(f"Final row count: {len(data)}")
print(data.isna().sum())

data.to_parquet("cleaned_small_data.parquet", index=False)
print("Saved.")

Final row count: 222735
rating               0
review_title         0
review_text          0
parent_asin          0
timestamp            0
helpful_vote         0
verified_purchase    0
category             0
product_title        0
average_rating       0
rating_number        0
features             0
store                0
dtype: int64
Saved.
